# Topic E — L5: Monte Carlo Foundations

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic 0 — Foundations](Topic-0_Foundations.ipynb) (§3 risk-neutral pricing, §5–6 GBM and BM), [Topic A](Topic-A_L1_Analytic-Approximations.ipynb) (Black-Scholes formula).

**What this notebook covers**
1. The Monte Carlo (MC) estimator — what it is, why it works
2. **CLT** convergence rate and standard errors
3. **Confidence intervals** — the working tool for MC results
4. Generating random variables — **inverse CDF** and **Box-Muller**
5. **Direct simulation** when payoff is a function of $S_T$ only (no path needed)
6. **Euler scheme** for SDEs that *do* need a path
7. **Milstein** (light touch) and convergence orders
8. **Correlated Brownian motions** via Cholesky
9. **Worked HW5 problem** — call on forward in an exponential OU model
10. Common pitfalls (wrong drift, mixing $r$ vs $\nu$, etc.)

**Exam map**

| Source | What it tests |
|---|---|
| L5.3–L5.6 | MC estimator, CLT, confidence intervals |
| L5.8–L5.9 | Inverse CDF, Box-Muller |
| L5.11 | Direct simulation of $S_T$ for GBM call |
| L5.14–L5.17 | Euler / Milstein schemes |
| L5.21 | Cholesky for correlated BMs |
| HW5 P1 | Full MC engine for exponential OU |
| Practice-Final P3 | Spot bugs in MC (see Topic F) |
| Practice-Final P5 | GBM drift mistake (see Topic H) |


In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

def bs_call(S, K, T, r, sigma):
    if T <= 0:
        return max(S - K, 0.0)
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / std
    d2 = d1 - std
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)


## 1. The Monte Carlo estimator (L5.3)

### The setup

By risk-neutral pricing (Topic 0 §3):
$$C = \mathbb E[Y], \qquad Y = e^{-\int_0^T r_s\,ds}\,\text{payoff}.$$

We **cannot** always compute the expectation analytically (e.g. Asian options, basket options, stochastic-vol models).

### The Monte Carlo estimator

1. Draw $Y_1, Y_2, \ldots, Y_M$ **independently and identically distributed** like $Y$.
2. Take the sample average:
$$\boxed{\widehat C_M := \frac{Y_1 + Y_2 + \cdots + Y_M}{M}.}$$

### Why it works (Law of Large Numbers)

By the **strong law of large numbers**, as $M \to \infty$:
$$\widehat C_M \;\xrightarrow{\text{a.s.}}\; \mathbb E[Y] = C.$$

**Plain English:** Draw enough independent samples and the average converges to the expectation.

### Why MC matters in option pricing

| Method | Best for |
|---|---|
| Closed form (BS) | Vanilla European with constant vol |
| Trees, FD | Americans, barriers, 1–2 risk factors |
| **Monte Carlo** | Anything else — Asians, baskets, stochastic vol, path-dependent, high-dimensional |

The cost: MC is **slow** — error shrinks only as $1/\sqrt M$.


## 2. Convergence rate: $1/\sqrt M$ (L5.4)

### Variance of the estimator

Let $\sigma^2 = \mathrm{Var}(Y)$ (here $\sigma$ is **not** stock volatility — it is the variance of the discounted payoff).

Because the $Y_m$ are IID:
$$\mathrm{Var}(\widehat C_M) = \frac{1}{M^2}\sum_{m=1}^M \mathrm{Var}(Y_m) = \frac{M\sigma^2}{M^2} = \frac{\sigma^2}{M}.$$

### Standard error

The **standard error** is the standard deviation of the estimator:
$$\text{SE} = \frac{\sigma}{\sqrt M}.$$

**Plain English:** To halve your error you need $4\times$ as many paths. To get one more decimal place you need $100\times$ as many paths.

### Central Limit Theorem (CLT)

For large $M$:
$$\frac{\widehat C_M - C}{\sigma/\sqrt M} \;\xrightarrow{d}\; \mathcal N(0,1).$$

We do **not** know $\sigma$ in practice, so we use the **sample standard deviation**:
$$\hat\sigma_M^2 := \frac{1}{M-1}\sum_{m=1}^M (Y_m - \widehat C_M)^2.$$

Replacing $\sigma$ with $\hat\sigma_M$ keeps the CLT result valid asymptotically.


## 3. Confidence intervals (L5.5–L5.6)

### Definition

The asymptotic 100(1−p)% confidence interval is
$$\Bigl[\widehat C_M - z_{1-p/2}\,\frac{\hat\sigma_M}{\sqrt M},\quad \widehat C_M + z_{1-p/2}\,\frac{\hat\sigma_M}{\sqrt M}\Bigr]$$
where $z_{1-p/2} = N^{-1}(1-p/2)$.

| Confidence | $z$ |
|---|---|
| 90% ($p=0.10$) | 1.645 |
| **95% ($p=0.05$)** | **1.96** |
| 99% ($p=0.01$) | 2.576 |

The **radius** is
$$R = z_{1-p/2}\,\frac{\hat\sigma_M}{\sqrt M}.$$

### A reality check (L5.6 example)

If $\hat\sigma = 20$ and $M = 10\,000$:
$$R = \frac{1.96 \times 20}{\sqrt{10000}} = 0.40.$$

To shrink this to $R=0.04$, you need $M = 1\,000\,000$.

**Lesson:** Brute-force MC is expensive. Topic F (variance reduction) will fix this.


In [2]:
rng = np.random.default_rng(seed=0)
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
M_values = [100, 1_000, 10_000, 100_000]

print(f"{'M':>10} | {'price':>8} | {'std err':>8} | {'95% radius':>10}")
print("-" * 50)
for M in M_values:
    Z = rng.standard_normal(M)
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    Y = np.exp(-r*T) * np.maximum(ST - K, 0.0)
    mean = Y.mean()
    se = Y.std(ddof=1) / np.sqrt(M)
    radius = 1.96 * se
    print(f"{M:>10} | {mean:>8.4f} | {se:>8.4f} | {radius:>10.4f}")

print(f"\nBlack-Scholes truth: {bs_call(S0, K, T, r, sigma):.4f}")
print("Note: radius shrinks like 1/sqrt(M). 10x more paths -> ~3.16x smaller radius.")


         M |    price |  std err | 95% radius
--------------------------------------------------
       100 |  11.3397 |   1.4687 |     2.8787
      1000 |   9.5332 |   0.4455 |     0.8731
     10000 |  10.5807 |   0.1470 |     0.2882
    100000 |  10.4264 |   0.0466 |     0.0912

Black-Scholes truth: 10.4506
Note: radius shrinks like 1/sqrt(M). 10x more paths -> ~3.16x smaller radius.


## 4. Generating random variables

We need a way to produce samples from any distribution we want. The starting point is always a stream of independent **Uniform(0,1)** draws (Python: `numpy.random.Generator.random()`).

### 4.1 Inverse CDF method (L5.8)

To draw $X$ with CDF $F$:
1. Draw $U \sim \mathrm{Uniform}(0,1)$.
2. Set $X := F^{-1}(U)$.

**Why it works:**
$$\mathbb P(X \le x) = \mathbb P(F^{-1}(U) \le x) = \mathbb P(U \le F(x)) = F(x). \checkmark$$

**Intuition:** Pick a percentile uniformly, then find the corresponding value.

**Example (normal):** Use `norm.ppf(U)`. Excel: `NORMSINV(RAND())`.

### 4.2 Box-Muller (L5.9)

If you need to roll your own normal generator from uniforms, Box-Muller is a cute trick:

Draw $U_1, U_2 \sim \mathrm{Uniform}(0,1)$ independently. Set
$$R := -2\log U_1, \qquad (X, Y) := \bigl(\sqrt R\cos(2\pi U_2),\; \sqrt R\sin(2\pi U_2)\bigr).$$
Then $X, Y$ are **independent standard normal**.

**Why it works:** If $X, Y$ are independent $\mathcal N(0,1)$, then $X^2 + Y^2 \sim \chi^2_2 = \mathrm{Exp}(1/2)$, and the angle is uniform on $[0, 2\pi)$. Box-Muller inverts this.

You probably will not need to code Box-Muller by hand on the exam — but recognize it.


In [3]:
rng = np.random.default_rng(seed=42)
N = 100_000

U1 = rng.random(N)
U2 = rng.random(N)
R = -2 * np.log(U1)
X = np.sqrt(R) * np.cos(2*np.pi*U2)
Y = np.sqrt(R) * np.sin(2*np.pi*U2)

print(f"Mean(X)   = {X.mean():.4f}  (should be 0)")
print(f"Std(X)    = {X.std():.4f}  (should be 1)")
print(f"Corr(X,Y) = {np.corrcoef(X, Y)[0,1]:.4f}  (should be 0)")


Mean(X)   = 0.0043  (should be 0)
Std(X)    = 1.0000  (should be 1)
Corr(X,Y) = 0.0037  (should be 0)


## 5. Direct simulation (no path needed) — GBM call (L5.11)

### When this works

If the payoff depends **only on $S_T$** (not on the whole path), and you can write $S_T$ as a known function of normal random variables, you do **not** need to simulate the path. Just simulate $S_T$ in one step.

### GBM example

For $dS_t = rS_t\,dt + \sigma S_t\,dW_t$ (Topic 0 §5), the closed-form solution is
$$S_T = S_0 \exp\Bigl((r - \tfrac12\sigma^2)T + \sigma\sqrt T\,Z\Bigr), \qquad Z \sim \mathcal N(0,1).$$

For a European call, the MC estimator is
$$\widehat C_M = \frac{1}{M}\sum_{m=1}^M e^{-rT}\bigl(S_T^{(m)} - K\bigr)^+,
\qquad S_T^{(m)} = S_0\exp\bigl((r-\tfrac12\sigma^2)T + \sigma\sqrt T\,Z^{(m)}\bigr).$$

### Exam pitfall — the drift adjustment

**Wrong:** `S0 * np.exp(r*T + sigma*np.sqrt(T)*Z)` ← Practice-Final P5 mistake.

**Right:** `S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)`.

The $-\tfrac12\sigma^2$ in the exponent is the **Itô correction** (Topic 0 §5). It is needed so that $\mathbb E[S_T] = S_0 e^{rT}$ — i.e. the forward price.

**Sanity check below:** the "wrong" formula gives the wrong mean.


In [4]:
rng = np.random.default_rng(seed=0)
S0, T, r, sigma = 100, 1.0, 0.05, 0.30
M = 200_000
Z = rng.standard_normal(M)

S_wrong = S0 * np.exp(r*T + sigma*np.sqrt(T)*Z)
S_right = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)

print(f"Theory:       E[S_T] = S0 * exp(rT) = {S0*np.exp(r*T):>10.4f}")
print(f"Wrong drift:  sample mean          = {S_wrong.mean():>10.4f}  <- biased")
print(f"Right drift:  sample mean          = {S_right.mean():>10.4f}  <- matches")


Theory:       E[S_T] = S0 * exp(rT) =   105.1271
Wrong drift:  sample mean          =   109.9806  <- biased
Right drift:  sample mean          =   105.1412  <- matches


## 6. Euler scheme — when you do need the whole path (L5.14)

### When you need a path

- Asian options ($\int_0^T S_t\,dt$ or $\frac{1}{T}\sum_n S_{t_n}$)
- Barrier options (hitting times)
- Local-vol models $dS_t = \sigma(S_t,t)S_t\,dW_t$
- Mean-reverting models — like the exponential OU in HW5

### Euler-Maruyama scheme

For SDE
$$dX_t = a(X_t, t)\,dt + b(X_t, t)\,dW_t,$$
discretize on a uniform grid $0 = t_0 < t_1 < \cdots < t_N = T$ with $\Delta t = T/N$. Define recursively:
$$\boxed{X^{(m)}_{t_{n+1}} = X^{(m)}_{t_n} + a(X^{(m)}_{t_n}, t_n)\,\Delta t + b(X^{(m)}_{t_n}, t_n)\,Z^{(m)}_n\sqrt{\Delta t}}$$
with $Z_n^{(m)} \sim \mathcal N(0,1)$ IID across $n, m$.

**Plain English:** At each step, take the deterministic drift forward by $\Delta t$, and add a normal increment scaled by $\sqrt{\Delta t}$ — that mirrors the Brownian increment $W_{t_{n+1}} - W_{t_n} \sim \mathcal N(0, \Delta t)$.

### Convergence orders (L5.15–L5.16)

| Type | What it measures | Euler order |
|---|---|---|
| **Weak** | $|\mathbb E f(X_T) - \mathbb E f(\widehat X_T)|$ | $O(\Delta t)$ |
| **Strong** | $\mathbb E\lvert X_T - \widehat X_T\rvert$ | $O(\Delta t^{1/2})$ |

For option pricing we only care about **weak** convergence (we average a function of $X_T$), so Euler is order 1 — good enough.

### Milstein scheme (L5.17, light touch)

Adds one term to capture the $b\,\partial b/\partial x$ contribution:
$$X^{(m)}_{t_{n+1}} = X^{(m)}_{t_n} + a\,\Delta t + b\,Z\sqrt{\Delta t} + \tfrac12\,b\,\tfrac{\partial b}{\partial x}\bigl(Z^2 - 1\bigr)\,\Delta t.$$
Strong order improves to 1; weak order stays at 1. **For option pricing it offers no improvement** in convergence rate. Mention it for completeness.


In [5]:
def euler_gbm_call(S0, K, T, r, sigma, N=100, M=100_000, seed=0):
    rng = np.random.default_rng(seed)
    dt = T / N
    Z = rng.standard_normal((M, N))
    logS = np.full(M, np.log(S0))
    nu = r - 0.5 * sigma**2  # under log dynamics, drift is nu
    for n in range(N):
        logS = logS + nu*dt + sigma*np.sqrt(dt)*Z[:, n]
    ST = np.exp(logS)
    Y = np.exp(-r*T) * np.maximum(ST - K, 0.0)
    return Y.mean(), Y.std(ddof=1)/np.sqrt(M)

S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
price_euler, se_euler = euler_gbm_call(S0, K, T, r, sigma, N=100, M=100_000)
truth = bs_call(S0, K, T, r, sigma)

print(f"Euler MC price (N=100, M=100k): {price_euler:.4f}  (SE {se_euler:.4f})")
print(f"Black-Scholes truth:            {truth:.4f}")
print(f"Difference:                     {price_euler - truth:+.4f}")
print()
print("Note: for GBM we could use the closed-form S_T directly (Section 5).")
print("Euler is illustrated here so the same code generalizes to non-GBM dynamics.")


Euler MC price (N=100, M=100k): 10.3827  (SE 0.0466)
Black-Scholes truth:            10.4506
Difference:                     -0.0679

Note: for GBM we could use the closed-form S_T directly (Section 5).
Euler is illustrated here so the same code generalizes to non-GBM dynamics.


## 7. Correlated Brownian motions via Cholesky (L5.18–L5.22)

### Why we need this

Many models have multiple risk factors:
- Basket options on correlated stocks
- Heston stochastic vol — stock $S$ and variance $V$ driven by correlated BMs
- Multi-currency models

You need to simulate $\bar W = (W^{[1]}, \ldots, W^{[D]})$ where $\mathrm{Corr}(\Delta W^{[i]}, \Delta W^{[j]}) = H_{ij}$ for some correlation matrix $H$.

### Cholesky decomposition

A symmetric matrix $H$ is a valid correlation matrix iff $H = LL^\top$ for some lower triangular $L$ (with diagonal entries $\ge 0$). Then if $Z$ is a vector of IID standard normals,
$$\bar Z := LZ$$
has covariance $\mathbb E[LZ(LZ)^\top] = L\,\mathbb E[ZZ^\top]\,L^\top = LL^\top = H.$ ✓

So:
$$\bar W_t = L W_t, \qquad \Delta \bar W_n = L Z_n \sqrt{\Delta t}.$$

### Example: 2D correlation $\rho$

$$H = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix} = LL^\top, \qquad L = \begin{pmatrix} 1 & 0 \\ \rho & \sqrt{1-\rho^2} \end{pmatrix}.$$

So
$$\bar Z^{[1]} = Z^{[1]}, \qquad \bar Z^{[2]} = \rho Z^{[1]} + \sqrt{1-\rho^2}\,Z^{[2]}.$$

This is the most common $2 \times 2$ formula you will use. Memorize it.

### Python tools

| Goal | Function |
|---|---|
| Find $L$ from $H$ | `np.linalg.cholesky(H)` |
| Draw $LZ$ directly | `rng.multivariate_normal(mean, H)` |

### Interview question (L5.20)

If $\mathrm{Corr}(X,Y) = \mathrm{Corr}(X,Z) = \mathrm{Corr}(Y,Z) = \rho$, what are the possible values of $\rho$?

<details><summary><b>Answer</b></summary>

$H$ must be positive semi-definite. The $3\times 3$ matrix has eigenvalues $1 + 2\rho$ (once) and $1 - \rho$ (twice). Both must be $\ge 0$:
$$1 + 2\rho \ge 0 \quad\text{and}\quad 1 - \rho \ge 0 \;\Longrightarrow\; \rho \in [-\tfrac12, 1].$$

Equivalently use $\det H = (1-\rho)^2(1+2\rho) \ge 0$.
</details>


In [6]:
rho = 0.6
H = np.array([[1.0, rho], [rho, 1.0]])
L = np.linalg.cholesky(H)
print(f"Target H:\n{H}")
print(f"Cholesky L:\n{L}")
print(f"L @ L.T  recovers H:\n{L @ L.T}")
print(f"Closed-form L for 2x2: [[1,0],[rho,sqrt(1-rho^2)]] = [[1,0],[{rho},{np.sqrt(1-rho**2):.4f}]]")

rng = np.random.default_rng(0)
M = 200_000
Z = rng.standard_normal((M, 2))
Zbar = Z @ L.T  # equivalently Z[:,0]=Z1, Z[:,1]=rho*Z1+sqrt(1-rho^2)*Z2
print(f"\nEmpirical correlation: {np.corrcoef(Zbar.T)[0,1]:.4f}  (target {rho})")


Target H:
[[1.  0.6]
 [0.6 1. ]]
Cholesky L:
[[1.  0. ]
 [0.6 0.8]]
L @ L.T  recovers H:
[[1.  0.6]
 [0.6 1. ]]
Closed-form L for 2x2: [[1,0],[rho,sqrt(1-rho^2)]] = [[1,0],[0.6,0.8000]]

Empirical correlation: 0.5984  (target 0.6)


## 8. Worked example — HW5 P1: Call on a forward in exponential OU

### Model

The log-spot price $X_t = \log S_t$ follows a **mean-reverting Ornstein-Uhlenbeck** process under the risk-neutral measure:
$$dX_t = \kappa(\alpha - X_t)\,dt + \sigma\,dW_t.$$

**Parameters:**
- $\kappa$ — speed of mean reversion (how fast it pulls back)
- $\alpha$ — long-run mean of $\log S$
- $\sigma$ — volatility of log price

**Key fact for this problem:** because the SDE for $X$ is linear with constant coefficients, $X_t \mid X_0$ is **Gaussian**, and there is a closed-form expression for the forward price $F_{T_1}$ given the time-$T_1$ state.

### Forward price formula

Given $X_{T_1}$, the forward price at $T_1$ for delivery at $T_2$ is
$$F_{T_1} = \exp\Bigl(e^{-\kappa\tau}X_{T_1} + (1 - e^{-\kappa\tau})\alpha + \tfrac{\sigma^2}{4\kappa}(1 - e^{-2\kappa\tau})\Bigr),
\qquad \tau = T_2 - T_1.$$

(Derivation: $X_{T_2} \mid X_{T_1}$ is Gaussian; $F_{T_1} = \mathbb E_{T_1}[S_{T_2}] = \mathbb E_{T_1}[e^{X_{T_2}}]$ is the log-normal expectation.)

### Call on the forward

Payoff at $T_1$: $(F_{T_1} - K)^+$. Time-0 price by risk-neutral pricing:
$$C = e^{-rT_1}\,\mathbb E[(F_{T_1} - K)^+].$$

### MC strategy

1. **Euler-simulate** $X$ from $0$ to $T_1$ in $N$ steps (the SDE has state-dependent drift, so no exact closed form for the path).
2. **Compute $F_{T_1}^{(m)}$** using the closed-form formula above — no further simulation needed beyond $T_1$ because $F_{T_1}$ is a known function of $X_{T_1}$.
3. **Average** the discounted payoff over $M$ paths.

### Delta via bumping ($\partial C/\partial S_0$)

Use a forward difference with **common random numbers** (same $Z$ matrix for both bumped and unbumped estimates):
$$\widehat\Delta \approx \frac{\widehat C(S_0 + \varepsilon) - \widehat C(S_0)}{\varepsilon}.$$

Common random numbers cancel most of the variance, giving a much more accurate delta than independent simulations.


In [7]:
class XOU:
    def __init__(self, kappa, alpha, sigma, S0, r):
        self.kappa, self.alpha, self.sigma = kappa, alpha, sigma
        self.S0, self.r = S0, r

class CallOnForwardPrice:
    def __init__(self, K1, T1, T2):
        self.K1, self.T1, self.T2 = K1, T1, T2

class MCengine:
    def __init__(self, N, M, epsilon, seed):
        self.N, self.M = N, M
        self.epsilon = epsilon
        self.rng = np.random.default_rng(seed=seed)

    def price_call_XOU(self, contract, dynamics):
        K, T1, T2 = contract.K1, contract.T1, contract.T2
        kappa, alpha, sigma = dynamics.kappa, dynamics.alpha, dynamics.sigma
        r, S0 = dynamics.r, dynamics.S0

        dt = T1 / self.N
        Z = self.rng.normal(size=(self.M, self.N))

        tau = T2 - T1
        a_coef = np.exp(-kappa*tau)
        b_coef = (1 - a_coef)*alpha + sigma**2/(4*kappa)*(1 - np.exp(-2*kappa*tau))

        def discounted_payoff(S_init):
            X = np.full(self.M, np.log(S_init))
            for n in range(self.N):
                X = X + kappa*(alpha - X)*dt + sigma*np.sqrt(dt)*Z[:, n]
            F_T1 = np.exp(a_coef*X + b_coef)
            return np.exp(-r*T1) * np.maximum(F_T1 - K, 0.0)

        Y0 = discounted_payoff(S0)
        Yeps = discounted_payoff(S0 + self.epsilon)

        call_price = Y0.mean()
        standard_error = Y0.std(ddof=1) / np.sqrt(self.M)
        call_delta = (Yeps.mean() - Y0.mean()) / self.epsilon

        return call_price, standard_error, call_delta

hw5dynamics = XOU(kappa=0.472, alpha=4.4, sigma=0.368, S0=106.9, r=0.05)
hw5contract = CallOnForwardPrice(K1=103.2, T1=0.5, T2=0.75)
hw5MC = MCengine(N=100, M=200_000, epsilon=0.01, seed=0)

price, se, delta = hw5MC.price_call_XOU(hw5contract, hw5dynamics)
print(f"Call price:        {price:.4f}")
print(f"Standard error:    {se:.4f}")
print(f"95% radius:        {1.96*se:.4f}")
print(f"Delta (bump):      {delta:.4f}")
print()
print("HW5 baseline: price 7.7880, SE 0.0298, delta 0.3415")


Call price:        7.7880
Standard error:    0.0298
95% radius:        0.0585
Delta (bump):      0.3415

HW5 baseline: price 7.7880, SE 0.0298, delta 0.3415


## 9. Common MC pitfalls (exam-critical)

### Pitfall 1 — wrong GBM drift

**Wrong:** `S0*exp(r*dt + sigma*sqrt(dt)*Z)`
**Right:** `S0*exp((r - 0.5*sigma**2)*dt + sigma*sqrt(dt)*Z)`

The $-\tfrac12\sigma^2$ is the Itô correction; without it $\mathbb E[S_T] \ne S_0 e^{rT}$.

### Pitfall 2 — `np.cumsum` for GBM paths

When building a path on a log grid, the cumulative drift is $\nu \cdot n \Delta t$ (a sum, **not** a single $\nu\Delta t$):
$$\log S_{t_n} = \log S_0 + \nu\,n\Delta t + \sigma\sqrt{\Delta t}\sum_{k=1}^n Z_k.$$
In code, use `np.cumsum(Z, axis=1)` for the noise and add $\nu\,\Delta t\cdot[1,2,3,\ldots]$ for drift (not just $\nu\,\Delta t$). Practice-Final P5 contains a version of this bug.

### Pitfall 3 — using `r` instead of `r - q` for dividend-paying stocks

If $S$ has dividend yield $q$, the risk-neutral drift is $r - q$, not $r$.

### Pitfall 4 — independent $Z$ in finite-difference delta

When computing $\widehat\Delta = (\widehat C(S_0+\varepsilon) - \widehat C(S_0))/\varepsilon$, **reuse the same $Z$ matrix** for both estimates. Independent $Z$'s explode the variance — see Topic G.

### Pitfall 5 — too-small $\varepsilon$ in bumping

The bias is $O(\varepsilon)$ but the variance scales like $1/\varepsilon^2$ (or $1/(M\varepsilon^2)$ — see Topic G). There is an optimal $\varepsilon$ that balances them.

### Pitfall 6 — using sample std vs population std for SE

Use `ddof=1` (sample std) in `np.std`. The denominator should be $M-1$, not $M$.

### Pitfall 7 — printing standard error vs radius

The exam typically asks for the **radius** of the confidence interval:
$$R = z_{1-p/2} \times \text{SE}.$$
For 95% this is $1.96\times \text{SE}$, not just SE.


## 10. Decision flowchart — does my problem need a path?

```mermaid
flowchart TD
    A[Need to price by MC] --> B{Payoff depends only on S_T?}
    B -->|Yes, e.g. European call| C{Can I sample S_T directly?}
    B -->|No, e.g. Asian/barrier| D[Need full path - use Euler L5.14]
    C -->|Yes - GBM, BS framework| E[Direct sampling L5.11<br/>One draw per path]
    C -->|No - state-dependent drift| D
    D --> F{Multiple correlated factors?}
    F -->|Yes| G[Cholesky L5.21<br/>Draw L*Z each step]
    F -->|No| H[Standard Euler]
```


## 11. Exam-style exercises

**E1.** A MC estimator with $M=40\,000$ paths gives sample mean $5.20$ and sample std $3.0$. Report the 95% confidence interval.

**E2.** Why is the MC error $O(1/\sqrt M)$ rather than $O(1/M)$?

**E3.** Write Box-Muller in two lines of code: given $U_1, U_2 \sim \mathrm{Uniform}(0,1)$, produce one standard normal.

**E4.** For 2D BM with correlation $\rho = 0.5$, write $\bar Z^{[1]}, \bar Z^{[2]}$ in terms of two IID standard normals $Z^{[1]}, Z^{[2]}$.

**E5.** In a MC call pricer, you replace `S0*exp((r - 0.5*sigma**2)*T + sigma*sqrt(T)*Z)` with `S0*exp(r*T + sigma*sqrt(T)*Z)`. Does the price come out too high or too low? Why?


### Solutions

<details><summary><b>E1</b></summary>

$\text{SE} = 3.0/\sqrt{40000} = 0.015$. Radius $= 1.96 \times 0.015 = 0.0294$. CI: $[5.20 - 0.0294,\; 5.20 + 0.0294] = [5.1706,\; 5.2294]$.
</details>

<details><summary><b>E2</b></summary>

Variance of the average of $M$ IID variables is $\sigma^2/M$. Standard deviation is $\sigma/\sqrt M$. Halving error costs 4× the work; one extra decimal place costs 100×.
</details>

<details><summary><b>E3</b></summary>

```python
Z = np.sqrt(-2*np.log(U1)) * np.cos(2*np.pi*U2)
```
(The $\sin$ version gives a second independent normal.)
</details>

<details><summary><b>E4</b></summary>

$\bar Z^{[1]} = Z^{[1]}$, $\bar Z^{[2]} = 0.5\,Z^{[1]} + \sqrt{1-0.25}\,Z^{[2]} = 0.5\,Z^{[1]} + \frac{\sqrt 3}{2}\,Z^{[2]}$.
</details>

<details><summary><b>E5</b></summary>

The mean of $S_T$ becomes $S_0 e^{rT + \sigma^2 T/2}$ instead of $S_0 e^{rT}$. So $S_T$ is biased **high**, payoffs are higher, and the call price comes out **too high**. (Concretely: the missing Itô correction shifts the log-mean up by $\tfrac12\sigma^2 T$.)
</details>


## 12. Cheat sheet — Topic E

| # | Fact | Use |
|---|---|---|
| 1 | $\widehat C_M = \frac{1}{M}\sum Y_m \to C = \mathbb E[Y]$ | MC estimator |
| 2 | $\mathrm{Var}(\widehat C_M) = \sigma^2/M$; $\text{SE} = \sigma/\sqrt M$ | Convergence rate |
| 3 | 95% CI radius $= 1.96\,\text{SE}$ | Reporting accuracy |
| 4 | $X = F^{-1}(U)$, $U\sim U(0,1)$ | Inverse CDF |
| 5 | $S_T = S_0 \exp((r-\tfrac12\sigma^2)T + \sigma\sqrt T Z)$ | GBM direct simulation |
| 6 | Euler: $X_{t_{n+1}} = X_{t_n} + a\Delta t + b Z\sqrt{\Delta t}$ | Path simulation |
| 7 | Euler weak order: $O(\Delta t)$; strong: $O(\sqrt{\Delta t})$ | Convergence |
| 8 | 2D Cholesky: $\bar Z_2 = \rho Z_1 + \sqrt{1-\rho^2} Z_2$ | Correlated BMs |
| 9 | Use **common random numbers** for FD delta | Variance reduction |
| 10 | Drift in log: $\nu = r - \tfrac12\sigma^2$ (no dividends) | Exam pitfall |

---

*Next:* **Topic F — L6: Variance Reduction** (control variates, antithetic, conditional MC, stratified sampling). Practice-Final P3 lives there.
